# Lecture: Finding Boxes — IoU, NMS, and Two Small Heads (Tiny Detector)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Compute IoU for two boxes and say what 0 and 1 mean (§4).
- Explain how NMS turns a pile of duplicate boxes into one box per object (§5).
- Read the system map (§9) and narrate one detection: image → two heads → NMS → kept boxes → scores.


## The big idea

Imagine two sticky notes on a whiteboard: the green one is where the object really is (the **ground-truth box**), the red one is where the model guessed. **IoU (Intersection over Union)** asks: "of all the whiteboard area covered by either note, how much is covered by *both*?" Identical notes score 1.0, non-touching notes score 0.0, and our worked example scores 0.2 (shared area 3.0 out of 15.0 total). A detector first predicts *many* overlapping guesses from two small output branches — a **box head** regressing 4 box numbers and a **class head** voting on the shape — then **NMS (non-maximum suppression)** plays the strict teacher: keep the highest-**confidence** box, delete every neighbour overlapping it too much, repeat until one clean box per object remains. Two summary numbers grade the whole system: **mIoU** (average overlap) and **precision@0.5** (fraction of guesses at least half-right), trained with **SmoothL1** (box penalty) plus **cross-entropy** (class penalty).


In [ ]:
import numpy as np  # bring in numpy for the box-area arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the box drawing
import matplotlib.patches as patches  # bring in patches to draw rectangles
np.random.seed(8)  # fix the seed so the picture is identical every run
gt = [1.0, 1.0, 4.0, 4.0]  # green truth box as [left, top, right, bottom]
pr = [2.5, 2.0, 5.5, 5.0]  # red predicted box as [left, top, right, bottom]
ix0 = max(gt[0], pr[0])  # overlap left edge: the larger of the two lefts
iy0 = max(gt[1], pr[1])  # overlap top edge: the larger of the two tops
ix1 = min(gt[2], pr[2])  # overlap right edge: the smaller of the two rights
iy1 = min(gt[3], pr[3])  # overlap bottom edge: the smaller of the two bottoms
inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)  # shared area: 1.5 wide x 2.0 tall = 3.0
area_g = (gt[2] - gt[0]) * (gt[3] - gt[1])  # green box area: 3 x 3 = 9.0
area_p = (pr[2] - pr[0]) * (pr[3] - pr[1])  # red box area: 3 x 3 = 9.0
union = area_g + area_p - inter  # total covered area: 9 + 9 - 3 = 15.0
iou = inter / union  # IoU: shared divided by total = 3/15 = 0.2
print('intersection =', round(inter, 3))  # expect 3.0: the orange overlap region
print('union =', round(union, 3))  # expect 15.0: both notes minus double-count
print('IoU =', round(iou, 4))  # expect 0.2: a weak guess, below the 0.5 bar
fig, ax = plt.subplots(figsize=(4, 4))  # make one square drawing panel
ax.set_xlim(0, 7)  # fix x range so boxes are not cropped
ax.set_ylim(0, 7)  # fix y range so boxes are not cropped
ax.add_patch(patches.Rectangle((gt[0], gt[1]), 3, 3, fill=False, edgecolor='lime', linewidth=3))  # draw the green truth box
ax.add_patch(patches.Rectangle((pr[0], pr[1]), 3, 3, fill=False, edgecolor='red', linewidth=3))  # draw the red guessed box
ax.fill([ix0, ix1, ix1, ix0], [iy0, iy0, iy1, iy1], color='orange', alpha=0.5)  # shade the shared region orange
ax.text(0.2, 6.4, f'IoU = {iou:.3f}', fontsize=12)  # write the IoU value on the plot
ax.set_title('Green = truth, red = guess, orange = shared')  # title the panel
ax.set_aspect('equal')  # keep squares square so areas look honest
fig.tight_layout()  # tidy spacing around the panel
plt.show()  # render the figure


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 measures overlap, §5 cleans up, §6–§8 build and grade the detector.

| Term | One-liner | Section |
|---|---|---|
| Bounding box (xyxy) | Four numbers (left, top, right, bottom) framing one object | §4 |
| IoU (Intersection over Union) | Overlap score: shared area ÷ total covered area (0 = apart, 1 = identical) | §4 |
| NMS (non-maximum suppression) | Cleanup: keep the best box, delete neighbours overlapping it too much | §5 |
| Confidence score | The model's 0-to-1 self-grade on each predicted box | §5 |
| Box head | The branch predicting the 4 box numbers (regression: *where*) | §6 |
| Class head | The branch predicting which shape the box holds (classification: *what*) | §6 |
| mIoU (mean IoU) | Average overlap over the validation set (toy below: 0.58) | §7 |
| Precision@0.5 | Fraction of predictions with IoU ≥ 0.5 (toy below: 0.60) | §7 |
| SmoothL1 loss | Gentle penalty for box-number errors (quadratic near 0, linear far away) | §8 |
| Cross-entropy loss | Penalty for probability placed on the wrong class | §8 |


### §4 Bounding boxes and IoU

**In one sentence: a bounding box is four numbers `[x0, y0, x1, y1]` framing one object, and IoU scores a guess by dividing the area both boxes share (intersection) by the area either covers (union) — 1.0 for identical boxes, 0.0 for boxes that never touch.**

The formula is pure geometry: `IoU = inter / (area_a + area_b − inter)`, where the subtraction removes the double-counted overlap. Our anchor pair shared 3.0 out of 15.0 for IoU 0.2 — below the lab's 0.5 correctness bar, so it would count as a miss. Watch it: the lab's own unit-test pair (offset by half a box: intersection 0.25, union 1.75) gives exactly 1/7 ≈ 0.1429, flanked by the two landmarks every detector engineer memorizes — self-overlap 1.0 and disjoint 0.0.


In [ ]:
import numpy as np  # reuse numpy for the IoU arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the landmark bars
np.random.seed(8)  # fix the seed for reproducibility
def iou_pair(a, b):  # IoU for two [x0, y0, x1, y1] boxes
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))  # overlap width, or 0 if apart
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))  # overlap height, or 0 if apart
    inter = ix * iy  # shared area
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter  # total covered area
    return inter / union if union > 0 else 0.0  # ratio, guarded against zero-area boxes
print('unit-test pair:', round(iou_pair([0, 0, 1, 1], [0.5, 0.5, 1.5, 1.5]), 6), '(= 1/7)')  # 0.25/1.75: the lab gate
print('identical boxes:', iou_pair([0, 0, 1, 1], [0, 0, 1, 1]))  # must be 1.0: perfect guess
print('disjoint boxes:', iou_pair([0, 0, 1, 1], [2, 2, 3, 3]))  # must be 0.0: no shared area
print('anchor pair:', round(iou_pair([1, 1, 4, 4], [2.5, 2.0, 5.5, 5.0]), 4))  # 0.2: same as the anchor picture
fig, ax = plt.subplots(figsize=(6, 3.5))  # make one panel for the three landmarks
ax.bar(['identical (1.0)', 'offset half-box (1/7)', 'disjoint (0.0)'], [1.0, 1 / 7, 0.0], color='steelblue')  # one bar per landmark
ax.axhline(0.5, color='red', linestyle='--', label='correctness bar 0.5')  # lab threshold: above = found, below = miss
ax.set_ylabel('IoU')  # label the overlap axis
ax.set_title('IoU landmarks: only the identical pair clears 0.5')  # title the takeaway
ax.legend()  # show what the red line means
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §5 NMS and confidence — from a pile of guesses to one box

**In one sentence: NMS sorts all predicted boxes by confidence score, keeps the surest one, deletes every remaining box overlapping it above threshold (IoU > 0.5), and repeats on the survivors until one clean box per object is left.**

Greedy is the key word: there is no voting or averaging — the 0.9 box simply eats its 0.8 and 0.7 neighbours because they overlap it at ~0.87, while the far-away 0.85 box survives untouched (its overlap with the winner is 0.0). Confidence is only a ranking tool here, not a truth meter: a wrong box can be confident, which is exactly why the lab still checks survivors against ground truth with IoU afterwards. Watch it: four boxes go in (`kept [0, 3]`, `suppressed [1, 2]`), two come out — the best of the pile plus the separate object.


In [ ]:
import numpy as np  # reuse numpy for score sorting and arrays
import matplotlib.pyplot as plt  # reuse pyplot for the before/after NMS panels
import matplotlib.patches as patches  # reuse patches to draw box rectangles
np.random.seed(8)  # fix the seed so box jitter is identical every run
boxes = np.array([[1., 1., 4., 4.],  # box 0: best guess of the pile, score 0.9
                  [1.2, 1.1, 4.1, 4.0],  # box 1: near-duplicate, score 0.8
                  [1.1, 0.9, 3.9, 4.2],  # box 2: another near-duplicate, score 0.7
                  [6., 6., 8., 8.]])  # box 3: far-away separate object, score 0.85
scores = np.array([0.9, 0.8, 0.7, 0.85])  # confidence self-grade per box (higher = surer)
def iou_pair(a, b):  # same IoU helper as §4
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))  # overlap width or 0
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))  # overlap height or 0
    inter = ix * iy  # shared area
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter  # union area
    return inter / ua if ua > 0 else 0.0  # IoU with zero-area guard
order = list(np.argsort(-scores))  # box indices sorted best-first: [0, 3, 1, 2]
keep = []  # accepted boxes that survive suppression
while order:  # loop until every box is kept or deleted
    i = order.pop(0)  # take the current highest-scoring box
    keep.append(i)  # keep it: it is a local winner
    order = [j for j in order if iou_pair(boxes[i], boxes[j]) <= 0.5]  # drop neighbours above threshold
print('scores:', list(scores))  # all four self-grades
print('kept indices:', keep)  # expect [0, 3]: pile winner plus the far object
print('suppressed:', [k for k in range(4) if k not in keep])  # expect [1, 2]: eaten duplicates
fig, ax = plt.subplots(1, 2, figsize=(8, 4))  # make before/after panels
for a in ax:  # configure both panels identically
    a.set_xlim(0, 9)  # fix x range for fair comparison
    a.set_ylim(0, 9)  # fix y range for fair comparison
    a.set_aspect('equal')  # keep box shapes honest on both panels
for k in range(4):  # draw all four raw boxes on the left
    x0, y0, x1, y1 = boxes[k]  # unpack one box's corners
    ax[0].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red'))  # red raw box
    ax[0].text(x0, y0 - 0.4, f'{scores[k]:.1f}', fontsize=9)  # label it with its confidence
ax[0].set_title('before NMS: 4 boxes, 2 objects')  # title the left panel
for k in keep:  # draw only survivors on the right
    x0, y0, x1, y1 = boxes[k]  # unpack one surviving box
    ax[1].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='green', linewidth=2))  # green kept box
    ax[1].text(x0, y0 - 0.4, f'{scores[k]:.1f}', fontsize=9)  # label survivor with its score
ax[1].set_title('after NMS: duplicates removed')  # title the right panel
fig.suptitle('NMS keeps the best box, kills overlapping neighbours')  # overall takeaway
fig.tight_layout()  # tidy spacing between the panels
plt.show()  # render the figure


### §6 Box head vs class head — where vs what

**In one sentence: the box head regresses 4 box numbers (answering *where is it*), while the class head predicts logits over the shape classes (answering *what is it*) — both read the same shared image embedding.**

Concretely the lab's backbone plus shared layer compress each 64×64×3 image into one 128-number embedding; the box head maps it through a 128→4 matrix plus sigmoid (sigmoid squashes each output into the legal [0, 1] normalized-box range), and the class head maps it through a 128→3 matrix into 3 shape logits. Watch it: our toy 8-number embedding predicts box ≈ [0.69, 0.37, 0.43, 0.73] and class votes ≈ [0.42, 0.19, 0.38] — an untrained toy head, so read the *shapes* (4 sigmoid numbers, 3 votes summing to 1, square ahead), not the values.


In [ ]:
import numpy as np  # reuse numpy for the head arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the two-heads diagram
np.random.seed(8)  # fix the seed for reproducibility
feat = np.array([0.5, -0.3, 0.8, 0.1, -0.6, 0.4, 0.2, -0.1])  # toy 8-number shared embedding (lab uses 128)
Wbox = np.array([[0.4, -0.2, 0.1, 0.3], [-0.5, 0.6, -0.1, 0.2], [0.3, 0.1, -0.4, 0.5], [0.2, -0.3, 0.6, -0.2], [-0.1, 0.4, 0.2, -0.5], [0.6, -0.1, -0.3, 0.4], [-0.4, 0.2, 0.5, 0.1], [0.1, 0.5, -0.2, -0.3]])  # toy box matrix 8->4
Wcls = np.array([[0.5, -0.3, 0.1], [-0.2, 0.6, -0.4], [0.4, 0.1, -0.5], [-0.3, -0.2, 0.6], [0.2, 0.5, -0.1], [-0.6, 0.3, 0.4], [0.1, -0.4, 0.2], [0.3, 0.2, -0.3]])  # toy class matrix 8->3
raw_box = feat @ Wbox  # box head: 4 raw numbers = embedding dot each column
box = 1 / (1 + np.exp(-raw_box))  # sigmoid: squash each into the legal [0, 1] box range
logits = feat @ Wcls  # class head: 3 raw votes, one per shape
e = np.exp(logits - logits.max())  # stable exponentiate for the class softmax
probs = e / e.sum()  # normalize: 3 class probabilities summing to 1
print('box head ->', np.round(box, 2))  # expect ~[0.69 0.37 0.43 0.73] (seed 8; rerun prints exact): a centered middle box
print('class head ->', np.round(probs, 2))  # plotted below: square wins with the top share
print('predicted class:', int(np.argmax(probs)), '(0 = square)')  # argmax of the votes
fig, ax = plt.subplots(figsize=(8, 3))  # make one wide panel for the fork diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 3)  # fix placement range
ax.text(1.5, 1.5, 'shared\nembedding\n(128)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))  # trunk box
ax.text(5, 2.1, 'box head: 4 numbers\nWHERE (sigmoid)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue'))  # where branch
ax.text(5, 0.7, 'class head: 3 logits\nWHAT (softmax)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))  # what branch
ax.text(8.5, 1.5, 'NMS (§5)\nmerge', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='plum'))  # merge box
ax.annotate('', xy=(3.6, 2.1), xytext=(2.6, 1.6), arrowprops=dict(arrowstyle='->'))  # trunk to box head
ax.annotate('', xy=(3.6, 0.7), xytext=(2.6, 1.4), arrowprops=dict(arrowstyle='->'))  # trunk to class head
ax.annotate('', xy=(7.7, 1.5), xytext=(6.6, 1.9), arrowprops=dict(arrowstyle='->'))  # box head to NMS
ax.annotate('', xy=(7.7, 1.5), xytext=(6.6, 0.9), arrowprops=dict(arrowstyle='->'))  # class head to NMS
ax.set_title('One embedding forks into WHERE and WHAT')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §7 mIoU and precision@0.5 — grading the detector

**In one sentence: mIoU is the average IoU over the validation set (how tight the boxes are on average), and precision@0.5 is the fraction of predictions reaching IoU ≥ 0.5 (how many guesses count as "found").**

The 0.5 bar is a convention, not physics: a guess overlapping half its truth is clearly the right object, slightly off — below it, too sloppy to trust. Watch it: five toy predictions `[0.81, 0.62, 0.45, 0.72, 0.30]` average to mIoU 0.58, and 3 of 5 clear the bar for precision@0.5 = 0.60 (the two red bars are the misses). The lab's trained net does better on both — roughly mIoU 0.69 with precision@0.5 near 0.90 — while its class accuracy is already 100%: finding *where* is the hard half of detection, naming *what* is nearly solved.


In [ ]:
import numpy as np  # reuse numpy for the metric arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the score bars
np.random.seed(8)  # fix the seed for reproducibility
ious = np.array([0.81, 0.62, 0.45, 0.72, 0.30])  # toy IoUs for 5 validation images
miou = float(ious.mean())  # mean IoU: average tightness
p50 = float((ious >= 0.5).mean())  # precision@0.5: fraction clearing the bar
print('toy IoUs:', list(ious))  # the five per-image scores
print('mIoU:', round(miou, 2))  # expect 0.58
print('precision@0.5:', round(p50, 2))  # expect 0.60: three of five clear 0.5
fig, ax = plt.subplots(figsize=(7, 4))  # make one panel for the score bars
cols = ['steelblue' if v >= 0.5 else 'tomato' for v in ious]  # blue = found, red = miss
ax.bar(['img0', 'img1', 'img2', 'img3', 'img4'], ious, color=cols)  # one bar per prediction
ax.axhline(0.5, color='red', linestyle='--', label='correctness bar 0.5')  # the pass/fail line
ax.axhline(miou, color='green', linestyle=':', label=f'mean = {miou:.2f}')  # dotted mean line
ax.set_ylim(0, 1)  # fix the valid IoU range
ax.set_ylabel('IoU vs truth')  # label the score axis
ax.set_title('mIoU 0.58, precision@0.5 = 0.60 (3 of 5 found)')  # title carries both metrics
ax.legend()  # show what the lines mean
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §8 SmoothL1 and cross-entropy — the two training signals

**In one sentence: SmoothL1 punishes box-number errors gently (squared when tiny, straight-line when large so one wild box can't explode training), and cross-entropy punishes probability placed on the wrong class.**

SmoothL1's split personality is deliberate: near zero it behaves like `0.5 × e²` (smooth, settles precisely — error 0.2 costs just 0.02), far away like `|e| − 0.5` (error 2.0 costs 1.5 instead of the 4.0 that plain squaring would charge). Cross-entropy is the familiar `-log(p of the true class)`: 0.105 when the model gives the truth 0.9, a brutal 2.303 at 0.1. The lab simply adds them — one number steering *where*, one steering *what*. Watch it: the curves cross where each penalty bites hardest.


In [ ]:
import numpy as np  # reuse numpy for the loss curves
import matplotlib.pyplot as plt  # reuse pyplot for the penalty curves
np.random.seed(8)  # fix the seed for reproducibility
e = np.linspace(0, 3, 200)  # box errors from perfect to 3 pixels off
smooth = np.where(e < 1, 0.5 * e ** 2, e - 0.5)  # SmoothL1: quadratic near 0, linear beyond 1
l2 = e ** 2  # plain squared error for comparison (explodes on outliers)
print('SmoothL1(0.2) =', round(float(0.5 * 0.2 ** 2), 3))  # expect 0.02: gentle on small misses
print('SmoothL1(2.0) =', round(float(2.0 - 0.5), 3), 'vs L2(2.0) = 4.0')  # 1.5 vs 4.0: outliers tamed
print('CE(true p=0.9) =', round(float(-np.log(0.9)), 3))  # expect 0.105: confident-and-right is cheap
print('CE(true p=0.1) =', round(float(-np.log(0.1)), 3))  # expect 2.303: confident-and-wrong hurts
p = np.linspace(0.05, 1.0, 200)  # true-class probabilities for the CE curve
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: box penalty + class penalty
ax[0].plot(e, smooth, label='SmoothL1 (lab box loss)')  # gentle-then-linear curve
ax[0].plot(e, l2, '--', alpha=0.6, label='plain L2 (outlier bomb)')  # steep comparison curve
ax[0].plot([0.2, 2.0], [0.02, 1.5], 'ro')  # mark the two printed landmarks
ax[0].set_xlabel('|box error|')  # label the mistake axis
ax[0].set_ylabel('penalty')  # label the cost axis
ax[0].set_title('Box loss: kind to outliers')  # title the left story
ax[0].legend(fontsize=8)  # identify the curves
ax[1].plot(p, -np.log(p), label='CE = -log(p_true)')  # steep-near-zero curve
ax[1].plot([0.9, 0.1], [-np.log(0.9), -np.log(0.1)], 'ro')  # mark the two printed landmarks
ax[1].set_xlabel('probability on the TRUE class')  # label: only this probability counts
ax[1].set_ylabel('penalty')  # label the cost axis
ax[1].set_title('Class loss: brutal when confidently wrong')  # title the right story
ax[1].legend(fontsize=8)  # identify the curve
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map (§9)

One detection is a pipeline where every § term holds one station. The 64×64×3 image enters the tiny CNN backbone, which forks into the **box head** (4 sigmoid numbers, §6) and the **class head** (3 logits, §6) — trained jointly by SmoothL1 + cross-entropy (§8). Raw predictions pile up, **NMS** keeps the confident survivors (§5), and each survivor is scored against truth with **IoU** (§4) into **mIoU / precision@0.5** (§7). The trace below replays the end of that line: three overlapping guesses at confidences 0.9/0.8/0.7 collapse to box 0, whose IoU against truth is 1.0 — a perfect detect-and-localize.


In [ ]:
import numpy as np  # import numpy for the seed habit (diagram uses no randomness)
import matplotlib.pyplot as plt  # only pyplot draws: this cell is a pure diagram
np.random.seed(8)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('image\n64x64x3', 1), ('box head §6\nclass head §6', 4), ('NMS §5\nkeep best', 7.5), ('kept boxes\nIoU §4', 10), ('mIoU/P@0.5 §7', 12.5)]  # pipeline stations
colors = ['wheat', 'lightblue', 'plum', 'lightgreen', 'steelblue']  # color per station type
for (name, xc), c in zip(stages, colors):  # draw each pipeline station
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor=c))  # rounded colored box around it
ax.text(7, 3.5, 'DETECT: heads guess (§6, §8) -> NMS cleans (§5) -> metrics grade (§4, §7)', ha='center', fontsize=9, fontweight='bold')  # banner line
for xa, xb in [(1.9, 3.0), (5.1, 6.5), (8.5, 9.3), (10.9, 11.8)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2), arrowprops=dict(arrowstyle='->'))  # forward arrow left -> right
ax.set_title('System map: one detection through every vocabulary term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the end-to-end numeric trace
np.random.seed(8)  # fix the seed (arrays below are hardcoded; habit kept)
truth = np.array([0.30, 0.30, 0.70, 0.70])  # ground-truth box, normalized (§4)
boxes = np.array([[0.30, 0.30, 0.70, 0.70],  # guess 0: dead-on, confidence 0.9
                  [0.32, 0.31, 0.71, 0.70],  # guess 1: near-duplicate, confidence 0.8
                  [0.31, 0.29, 0.69, 0.72]])  # guess 2: near-duplicate, confidence 0.7
scores = np.array([0.9, 0.8, 0.7])  # confidence per guess (§5)
def iou_pair(a, b):  # same IoU helper as §4
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))  # overlap width or 0
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))  # overlap height or 0
    inter = ix * iy  # shared area
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter  # union area
    return inter / ua if ua > 0 else 0.0  # IoU with zero-area guard
order = list(np.argsort(-scores))  # best-first order (§5)
keep = []  # survivors
while order:  # greedy NMS loop (§5)
    i = order.pop(0)  # take the current winner
    keep.append(i)  # keep it
    order = [j for j in order if iou_pair(boxes[i], boxes[j]) <= 0.5]  # suppress close dupes
print('NMS kept:', keep, '(box 0 eats boxes 1 and 2)')  # expect [0]: one survivor
print('winner IoU vs truth:', round(iou_pair(boxes[keep[0]], truth), 4))  # expect 1.0: dead-on guess
print('verdict: IoU 1.0 >= 0.5 -> counts as found (§7)')  # the metric's reading of this trace


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, on synthetic shape photos:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `np.random.seed` habit as this lecture.
2. **Cell 2 (synthetic shapes)** — renders 1500 tiny 64×64 images, each with one colored rectangle (square / tall / wide) plus its true box and class — a sandbox where detection mistakes are easy to see.
3. **Cell 3 (IoU + NMS from scratch)** — the lab's `iou_pair`, `iou_matrix`, and greedy `nms` are exactly the §4 overlap math and §5 cleanup loop; unit tests check IoU = 1/7 on the offset pair and that NMS keeps boxes `[0, 2]` on duplicates.
4. **Cell 4 (tiny detector + training)** — a small CNN splits into a box head (4 sigmoid numbers, §6) and a class head (3 logits, §6), trained 12 epochs on SmoothL1 + cross-entropy (§8).
5. **Cell 5 (evaluation)** — mean IoU (~0.69), precision@0.5 (~0.90), and 100% class accuracy on 300 val images (§7); `results/boxes.png` overlays green truth vs red predictions exactly like the anchor plot.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Two boxes overlap with intersection 1 and union 7. What is the IoU, and is it a good detection at threshold 0.5?**

*Answer: IoU = 1/7 ≈ 0.143 (§4) — no, it is below 0.5 and counts as a miss (this is literally the lab's unit test; only the identical-pair landmark at 1.0 clears the bar in the §4 chart).*

2. **NMS gets 3 near-identical boxes at 0.9/0.8/0.7 plus a far box at 0.85. What survives?**

*Answer: the 0.9 box and the far 0.85 box (§5) — the winner is kept first and eats both neighbours above IoU 0.5, while the far box overlaps nothing and survives untouched. Rerun the §5 cell and read `kept [0, 3]`.*

3. **What does each of the lab's two heads predict, and what trains each?**

*Answer: the box head regresses 4 normalized box numbers trained by SmoothL1, the class head predicts 3 shape logits trained by cross-entropy (§6, §8) — *where* plus *what*, graded jointly by §7's metrics.*
